# Ticker search: alternative data for any company or commodity
Type a ticker or a commodity and get one page, like Yahoo Finance for alternative data: price, every linked prediction
market on Kalshi and Polymarket, the gap and the next event, analyst consensus (companies) or the futures curve
(commodities), linked assets, the history of each satellite index, the supply chain map with live alerts and attention
on X. Any signal can be turned into a monitor. Ten names are covered: NVIDIA, Brent, WTI, copper, gold, corn, wheat,
soybeans, coffee and sugar.

In [2]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from altinfo import gap, satellite, earnings, scoring, charts, valuation, eventbeta as eb
from altinfo.charts import NAVY, BLUE, RED, GREY
charts.style()
RAW = ROOT / "data" / "raw"
SNAP = json.load(open(ROOT / "data" / "snapshot_2026-11-09.json"))
LIVE = (RAW / "prices.csv").exists()
print("Mode:", "LIVE (data/raw)" if LIVE else "SNAPSHOT (figures of 9 Nov 2026; run scripts/fetch_data.py for live data)")
def kalshi_live(series):
    """Open Kalshi markets of a series with a mid price (bid/ask midpoint, else last trade)."""
    km = pd.read_csv(RAW / "kalshi_markets.csv")
    m = km[km["series"] == series].copy()
    mid = (m["yes_bid_dollars"] + m["yes_ask_dollars"]) / 2
    tight = (m["yes_ask_dollars"] - m["yes_bid_dollars"]).le(0.10) & m["yes_ask_dollars"].gt(0)
    m["mid"] = mid.where(tight, m["last_price_dollars"])
    return m
def poly_live(pattern):
    pm = pd.read_csv(RAW / "polymarket_markets.csv")
    return pm[pm.question.str.contains(pattern, case=False, na=False, regex=True) & (pm.closed != True)].sort_values("volume", ascending=False)
def live_calls(today="2026-11-10"):
    """The calls registered in the reports, with the market probability updated to the latest live data."""
    calls = pd.DataFrame(SNAP["calls"]).assign(asset=["NVDA", "BRENT"], date=["2026-11-17", "2026-11-30"],
                                                confidence="medium", date_asof=pd.Timestamp(today))
    calls["market_week_ago"] = [SNAP["nvda"]["real"]["polymarket_beat_history"]["2026-11-03"], np.nan]
    if LIVE:
        cd = pd.read_csv(RAW / "kalshi_candles.csv", parse_dates=["date"])
        cd = cd[cd.ticker.str.startswith("KXBRENTMON-26NOV30")].copy()
        cd["strike"] = cd.ticker.str.extract(r"-T([\d.]+)$")[0].astype(float)
        daily = cd.dropna(subset=["price"]).groupby("date").apply(lambda d: gap.ladder_prob_above(d.strike, d.price, 96), include_groups=False)
        bm = kalshi_live("KXBRENTMON").dropna(subset=["floor_strike", "mid"])
        calls.loc[calls.asset == "BRENT", "market"] = gap.ladder_prob_above(bm.floor_strike, bm.mid, 96)
        calls.loc[calls.asset == "BRENT", "market_week_ago"] = daily.loc[:pd.Timestamp(today) - pd.Timedelta(days=7)].iloc[-1]
        nb_ = poly_live(r"N[Vv][Ii][Dd][Ii][Aa].*beat|NVDA.*beat")
        if len(nb_):
            calls.loc[calls.asset == "NVDA", "market"] = float(nb_["price"].iloc[0])
    return calls
def load_prices():
    p = pd.read_csv(RAW / "prices.csv")
    p["date"] = pd.to_datetime(p["date"], format="mixed").dt.normalize()
    return p.pivot_table(index="date", columns="ticker", values="close").sort_index()

Mode: LIVE (data/raw)


In [3]:
from altinfo.search import ticker_page
px = load_prices() if LIVE else pd.DataFrame()
km = pd.read_csv(RAW / "kalshi_markets.csv") if LIVE else None
pm = pd.read_csv(RAW / "polymarket_markets.csv") if LIVE else None
calls = live_calls()
from altinfo import tiers
def show(query, tier="Institutional"):
    from altinfo.search import to_html
    page = ticker_page(query, px, km, pm, calls=calls, today="2026-11-10", root=ROOT)
    display(HTML(to_html(tiers.ticker_page_for(page, tier), query)))
show("NVDA")

Venue,Market,Contract,Probability,Volume,Closes
Kalshi,What will NVIDIA Corporation say during their next earnings call?,Gaming,44%,"7,081",2026-12-02
Kalshi,"What will Jensen Huang say during NVIDIA GTC Washington, D.C. Keynote originally scheduled for December 1, 2026?",Humanoid,36%,"2,739",2026-12-15
Kalshi,What will NVIDIA Corporation say during their next earnings call?,Rubin (3+ times),94%,"2,629",2026-12-02
Kalshi,What will NVIDIA Corporation say during their next earnings call?,Hyperscaler,94%,"2,257",2026-12-02
Kalshi,"What will Jensen Huang say during NVIDIA GTC Washington, D.C. Keynote originally scheduled for December 1, 2026?",Omniverse,57%,"1,537",2026-12-15
Kalshi,What will NVIDIA Corporation say during their next earnings call?,Taiwan,46%,"1,209",2026-12-02
Kalshi,"What will Jensen Huang say during NVIDIA GTC Washington, D.C. Keynote originally scheduled for December 1, 2026?",China / Chinese,31%,901,2026-12-15
Kalshi,What will NVIDIA Corporation say during their next earnings call?,Automation,20%,801,2026-12-02
Kalshi,"What will Jensen Huang say during NVIDIA GTC Washington, D.C. Keynote originally scheduled for December 1, 2026?",Event does not qualify,38%,504,2026-12-15
Kalshi,"What will Jensen Huang say during NVIDIA GTC Washington, D.C. Keynote originally scheduled for December 1, 2026?",Physical AI,52%,488,2026-12-15


**How to read it.** The page a client sees after typing a ticker. Top: price and recent moves. Left: every Kalshi and Polymarket contract linked to the name, with the probability the market gives it. Right: the gap and the next event, analyst consensus, the assets that move with it (corr is the correlation of daily returns over a year: 1 means they move together) and the signals we track, each of which can be turned into a monitor. Below: the history of each satellite index, the supply chain map with live alerts and attention on X.

In [4]:
show("Brent")

Venue,Market,Contract,Probability,Volume,Closes
Kalshi,"Will the 7-day moving average of transit calls through the Strait of Hormuz as reported by the IMF PortWatch be above 60 before January 1, 2027?","Before Jan 1, 2027",8%,"3,923,140",2027-01-05
Kalshi,"Will the 7-day moving average of transit calls through the Strait of Hormuz as reported by the IMF PortWatch be above 60 before December 1, 2026?","Before Dec 1, 2026",2%,"2,181,376",2026-12-01
Kalshi,"Will the 7-day moving average of transit calls through the Strait of Hormuz as reported by the IMF PortWatch be above 60 before July 1, 2027?","Before Jul 1, 2027",40%,"1,665,323",2027-07-01
Kalshi,"Will the 7-day moving average of transit calls through the Strait of Hormuz as reported by the IMF PortWatch be above 60 before April 1, 2027?","Before Apr 1, 2027",22%,"879,479",2027-04-01
Kalshi,"Will the 7-day moving average of transit calls through the Strait of Hormuz as reported by the IMF PortWatch be above 60 before January 1, 2028?","Before Jan 1, 2028",56%,"92,656",2028-01-01
Kalshi,"Will the 7-day moving average of transit calls through the Strait of Hormuz as reported by the IMF PortWatch be above 60 before November 1, 2027?","Before Nov 1, 2027",48%,"81,560",2027-11-02
Kalshi,"Will the 7-day moving average of transit calls through the Strait of Hormuz as reported by the IMF PortWatch be above 60 before January 1, 2029?","Before Jan 1, 2029",70%,"65,659",2029-01-01
Kalshi,"Will the 7-day moving average of transit calls through the Strait of Hormuz as reported by the IMF PortWatch be above 60 before July 1, 2028?","Before Jul 1, 2028",66%,"56,000",2028-07-01
Kalshi,"Will the 7-day moving average of transit calls through the Strait of Hormuz as reported by the IMF PortWatch be above 60 before June 1, 2027?","Before Jun 1, 2027",34%,"30,710",2027-06-08
Kalshi,"Will the 7-day moving average of transit calls through the Strait of Hormuz as reported by the IMF PortWatch be above 60 before February 1, 2027?","Before Feb 1, 2027",13%,"29,825",2027-02-02


**How to read it.** The same page for Brent crude: most linked markets are about Hormuz traffic and the US Iran talks, which drive the price.

In [5]:
show("copper")

Venue,Market,Contract,Probability,Volume,Closes
Kalshi,"Will the copper close price be above 6.57 USD/Lbs on November 30, 2026 at 5:00 PM EST?",Above $6.57,29%,965,2026-11-30
Kalshi,"Will the copper close price be above 6.39 USD/Lbs on November 30, 2026 at 5:00 PM EST?",Above $6.39,50%,253,2026-11-30
Kalshi,"Will the copper close price be above 6.51 USD/Lbs on November 30, 2026 at 5:00 PM EST?",Above $6.51,35%,187,2026-11-30
Kalshi,"Will the copper close price be above 6.27 USD/Lbs on November 30, 2026 at 5:00 PM EST?",Above $6.27,65%,162,2026-11-30
Kalshi,"Will the copper close price be above 6.45 USD/Lbs on November 30, 2026 at 5:00 PM EST?",Above $6.45,43%,125,2026-11-30
Kalshi,"Will the copper close price be above 6.44 USD/Lbs on November 13, 2026 at 5:00 PM EST?",Above $6.44,38%,86,2026-11-13
Kalshi,"Will the copper close price be above 6.15 USD/Lbs on November 30, 2026 at 5:00 PM EST?",Above $6.15,78%,85,2026-11-30
Kalshi,"Will the copper close price be above 6.29 USD/Lbs on November 13, 2026 at 5:00 PM EST?",Above $6.29,78%,76,2026-11-13
Kalshi,"Will the copper close price be above 6.41 USD/Lbs on November 13, 2026 at 5:00 PM EST?",Above $6.41,46%,52,2026-11-13
Kalshi,"Will the copper close price be above 5.69 USD/Lbs on November 13, 2026 at 5:00 PM EST?",Above $5.69,98%,47,2026-11-13


**How to read it.** The same page for copper: Kalshi month-end price contracts and the mining stocks most exposed to the copper price.

## Agriculture
Corn, wheat, soybeans, coffee and sugar. Physical evidence: rain and temperature in the key growing regions against the
10 year normal (Open-Meteo, free) and crop condition from imagery (paid, not in the prototype). Expectations: futures
prices; Kalshi listed crop contracts in April 2026 and withdrew them, so they appear only if they are relisted.

In [7]:
show("corn")

Place,Role,Country
Iowa,Growing region,United States
Illinois,Growing region,United States
Mato Grosso,Growing region,Brazil
Poltava,Growing region,Ukraine
New Orleans,Export port,United States


**How to read it.** Corn: the map shows the growing regions and the export port; the weather table compares rain over the last 30 days with the usual amount for the time of year. Very dry or very wet regions are what moves crop prices.

In [8]:
show("coffee")

Place,Role,Country
Sul de Minas,Growing region,Brazil
Dak Lak,Growing region,Vietnam
Huila,Growing region,Colombia
Santos,Export port,Brazil


**How to read it.** Coffee: the Brazilian, Vietnamese and Colombian growing regions. Dry weather in Minas Gerais during flowering is the classic driver of coffee price spikes.

## The same page in the free tier
No live satellite readings, no supply chain alerts and no gap score; prices, prediction markets, attention and the
calendar stay visible.

In [10]:
show("NVDA", tier="free")

Venue,Market,Contract,Probability,Volume,Closes
Kalshi,What will NVIDIA Corporation say during their next earnings call?,Gaming,44%,"7,081",2026-12-02
Kalshi,"What will Jensen Huang say during NVIDIA GTC Washington, D.C. Keynote originally scheduled for December 1, 2026?",Humanoid,36%,"2,739",2026-12-15
Kalshi,What will NVIDIA Corporation say during their next earnings call?,Rubin (3+ times),94%,"2,629",2026-12-02
Kalshi,What will NVIDIA Corporation say during their next earnings call?,Hyperscaler,94%,"2,257",2026-12-02
Kalshi,"What will Jensen Huang say during NVIDIA GTC Washington, D.C. Keynote originally scheduled for December 1, 2026?",Omniverse,57%,"1,537",2026-12-15
Kalshi,What will NVIDIA Corporation say during their next earnings call?,Taiwan,46%,"1,209",2026-12-02
Kalshi,"What will Jensen Huang say during NVIDIA GTC Washington, D.C. Keynote originally scheduled for December 1, 2026?",China / Chinese,31%,901,2026-12-15
Kalshi,What will NVIDIA Corporation say during their next earnings call?,Automation,20%,801,2026-12-02
Kalshi,"What will Jensen Huang say during NVIDIA GTC Washington, D.C. Keynote originally scheduled for December 1, 2026?",Event does not qualify,38%,504,2026-12-15
Kalshi,"What will Jensen Huang say during NVIDIA GTC Washington, D.C. Keynote originally scheduled for December 1, 2026?",Physical AI,52%,488,2026-12-15


**How to read it.** The free version of the NVIDIA page: locked boxes mark what only institutional clients see.